In [2]:
import os
import numpy as np
import jax
import jax.numpy as jnp

# ============================================================
# settings
# ============================================================
USE_X64 = True
jax.config.update("jax_enable_x64", USE_X64)
DTYPE = jnp.float64 if USE_X64 else jnp.float32
EPS = DTYPE(1e-12)

# ----------------------------
# paths
# ----------------------------
npz_path = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/sqp_dataset.npz"
theta_path = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/theta_sqp_coupled_fixedbatch_jitter1.npy"

# ----------------------------
# THIS matches your pasted training code
# layer_sizes = [3] + [30]*3 + [3]
# ----------------------------
hidden_dim = 30
num_hidden = 3
layer_sizes = [3] + [hidden_dim] * num_hidden + [3]

# ----------------------------
# geometry
# ----------------------------
Ls = DTYPE(0.4)
Lf = DTYPE(0.6)
Ly = DTYPE(1.0)
T_end = DTYPE(1.0)

x_min = DTYPE(0.0)
x_max = DTYPE(Ls + Lf)
y_min = DTYPE(0.0)
y_max = DTYPE(Ly)
t_min = DTYPE(0.0)
t_max = DTYPE(T_end)

# ============================================================
# load reference
# ============================================================
data = np.load(npz_path, allow_pickle=True)
print("keys:", data.files)

x = np.array(data["x"], dtype=np.float64)
y = np.array(data["y"], dtype=np.float64)

if "time" in data:
    time_arr = np.array(data["time"], dtype=np.float64)
else:
    time_arr = np.array(data["t"], dtype=np.float64)

T_ref = np.array(data["T"], dtype=np.float64)
phi_ref = np.array(data["phi"], dtype=np.float64)

print("x shape:", x.shape)
print("y shape:", y.shape)
print("time shape:", time_arr.shape)
print("T_ref shape:", T_ref.shape)
print("phi_ref shape:", phi_ref.shape)

nt, nx, ny = T_ref.shape

# ============================================================
# network helpers
# ============================================================
def normalize_xyt(X):
    xx = X[:, 0:1]
    yy = X[:, 1:2]
    tt = X[:, 2:3]

    x_n = DTYPE(2.0) * (xx - x_min) / (x_max - x_min + EPS) - DTYPE(1.0)
    y_n = DTYPE(2.0) * (yy - y_min) / (y_max - y_min + EPS) - DTYPE(1.0)
    t_n = DTYPE(2.0) * (tt - t_min) / (t_max - t_min + EPS) - DTYPE(1.0)
    return jnp.concatenate([x_n, y_n, t_n], axis=1)

def unflatten_params(theta, layer_sizes):
    params = []
    idx = 0
    for m, n in zip(layer_sizes[:-1], layer_sizes[1:]):
        w_size = m * n
        b_size = n
        W = theta[idx:idx + w_size].reshape((m, n))
        idx += w_size
        b = theta[idx:idx + b_size].reshape((n,))
        idx += b_size
        params.append({"W": W, "b": b})
    if idx != theta.size:
        raise ValueError(f"Did not consume all theta entries: used {idx}, total {theta.size}")
    return params

def mlp_apply(params, X):
    h = normalize_xyt(X)
    for i, layer in enumerate(params):
        h = h @ layer["W"] + layer["b"]
        if i < len(params) - 1:
            h = jnp.tanh(h)
    # outputs = [phi, Ts, Tf]
    return h

# ============================================================
# load theta
# ============================================================
theta_np = np.load(theta_path)
print("theta raw shape:", theta_np.shape)
print("theta size:", theta_np.size)

params = unflatten_params(jnp.array(theta_np, dtype=DTYPE), layer_sizes)

# ============================================================
# helper
# ============================================================
def rel_l2(pred_arr, true_arr):
    return np.linalg.norm(pred_arr - true_arr) / (np.linalg.norm(true_arr) + 1e-12)

# ============================================================
# final-time relative L2
# ============================================================
t_final = float(time_arr[-1])

XX, YY = np.meshgrid(x, y, indexing="ij")
TT = np.full_like(XX, t_final, dtype=np.float64)

X_eval = np.stack([XX.reshape(-1), YY.reshape(-1), TT.reshape(-1)], axis=1)
pred = np.array(mlp_apply(params, jnp.array(X_eval, dtype=DTYPE)))

phi_pred = pred[:, 0].reshape(nx, ny)
Ts_pred  = pred[:, 1].reshape(nx, ny)
Tf_pred  = pred[:, 2].reshape(nx, ny)

phi_true = phi_ref[-1]
T_true   = T_ref[-1]

fuel_mask = x < float(Ls) - 1e-12
cool_mask = x > float(Ls) + 1e-12

phi_rel = rel_l2(phi_pred[fuel_mask, :], phi_true[fuel_mask, :])
Ts_rel  = rel_l2(Ts_pred[fuel_mask, :],  T_true[fuel_mask, :])
Tf_rel  = rel_l2(Tf_pred[cool_mask, :],  T_true[cool_mask, :])

print("\nFINAL-TIME RELATIVE L2")
print(f"phi (fuel): {phi_rel:.6e}")
print(f"Ts  (fuel): {Ts_rel:.6e}")
print(f"Tf  (cool): {Tf_rel:.6e}")

# ============================================================
# all-time mean relative L2
# ============================================================
phi_rel_ts = []
Ts_rel_ts = []
Tf_rel_ts = []

for k, tk in enumerate(time_arr):
    TTk = np.full_like(XX, float(tk), dtype=np.float64)
    X_eval_k = np.stack([XX.reshape(-1), YY.reshape(-1), TTk.reshape(-1)], axis=1)
    pred_k = np.array(mlp_apply(params, jnp.array(X_eval_k, dtype=DTYPE)))

    phi_pred_k = pred_k[:, 0].reshape(nx, ny)
    Ts_pred_k  = pred_k[:, 1].reshape(nx, ny)
    Tf_pred_k  = pred_k[:, 2].reshape(nx, ny)

    phi_true_k = phi_ref[k]
    T_true_k   = T_ref[k]

    phi_rel_ts.append(rel_l2(phi_pred_k[fuel_mask, :], phi_true_k[fuel_mask, :]))
    Ts_rel_ts.append(rel_l2(Ts_pred_k[fuel_mask, :],   T_true_k[fuel_mask, :]))
    Tf_rel_ts.append(rel_l2(Tf_pred_k[cool_mask, :],   T_true_k[cool_mask, :]))

print("\nALL-TIME MEAN RELATIVE L2")
print(f"phi (fuel): {np.mean(phi_rel_ts):.6e}")
print(f"Ts  (fuel): {np.mean(Ts_rel_ts):.6e}")
print(f"Tf  (cool): {np.mean(Tf_rel_ts):.6e}")

keys: ['t', 'x', 'y', 'phi', 'T', 'phi_fuel', 'Ts', 'Tf', 'T_interface']
x shape: (513,)
y shape: (513,)
time shape: (1001,)
T_ref shape: (1001, 513, 513)
phi_ref shape: (1001, 513, 513)
theta raw shape: (2073,)
theta size: 2073

FINAL-TIME RELATIVE L2
phi (fuel): 2.239303e-03
Ts  (fuel): 7.319812e-02
Tf  (cool): 3.665019e-01

ALL-TIME MEAN RELATIVE L2
phi (fuel): 1.337153e+08
Ts  (fuel): 1.742826e+07
Tf  (cool): 1.631590e+07


In [3]:
import numpy as np
import jax
import jax.numpy as jnp

# ============================================================
# settings
# ============================================================
USE_X64 = True
jax.config.update("jax_enable_x64", USE_X64)
DTYPE = jnp.float64 if USE_X64 else jnp.float32
EPS = DTYPE(1e-12)

# paths
npz_path = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/sqp_dataset.npz"
theta_path = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/theta_sqp_coupled_fixedbatch_jitter1.npy"

# architecture from your training code
hidden_dim = 30
num_hidden = 3
layer_sizes = [3] + [hidden_dim] * num_hidden + [3]

# geometry
Ls = DTYPE(0.4)
Lf = DTYPE(0.6)
Ly = DTYPE(1.0)
T_end = DTYPE(1.0)

x_min = DTYPE(0.0)
x_max = DTYPE(Ls + Lf)
y_min = DTYPE(0.0)
y_max = DTYPE(Ly)
t_min = DTYPE(0.0)
t_max = DTYPE(T_end)

# ============================================================
# load reference
# ============================================================
data = np.load(npz_path, allow_pickle=True)

x = np.array(data["x"], dtype=np.float64)
y = np.array(data["y"], dtype=np.float64)
time_arr = np.array(data["t"], dtype=np.float64) if "t" in data else np.array(data["time"], dtype=np.float64)
T_ref = np.array(data["T"], dtype=np.float64)
phi_ref = np.array(data["phi"], dtype=np.float64)

nt, nx, ny = T_ref.shape

# ============================================================
# network helpers
# ============================================================
def normalize_xyt(X):
    xx = X[:, 0:1]
    yy = X[:, 1:2]
    tt = X[:, 2:3]

    x_n = DTYPE(2.0) * (xx - x_min) / (x_max - x_min + EPS) - DTYPE(1.0)
    y_n = DTYPE(2.0) * (yy - y_min) / (y_max - y_min + EPS) - DTYPE(1.0)
    t_n = DTYPE(2.0) * (tt - t_min) / (t_max - t_min + EPS) - DTYPE(1.0)
    return jnp.concatenate([x_n, y_n, t_n], axis=1)

def unflatten_params(theta, layer_sizes):
    params = []
    idx = 0
    for m, n in zip(layer_sizes[:-1], layer_sizes[1:]):
        w_size = m * n
        b_size = n
        W = theta[idx:idx + w_size].reshape((m, n))
        idx += w_size
        b = theta[idx:idx + b_size].reshape((n,))
        idx += b_size
        params.append({"W": W, "b": b})
    if idx != theta.size:
        raise ValueError(f"Did not consume all theta entries: used {idx}, total {theta.size}")
    return params

def mlp_apply(params, X):
    h = normalize_xyt(X)
    for i, layer in enumerate(params):
        h = h @ layer["W"] + layer["b"]
        if i < len(params) - 1:
            h = jnp.tanh(h)
    # outputs = [phi, Ts, Tf]
    return h

# ============================================================
# load theta
# ============================================================
theta_np = np.load(theta_path)
theta = jnp.array(theta_np, dtype=DTYPE)
params = unflatten_params(theta, layer_sizes)

# ============================================================
# predict on whole space-time grid
# ============================================================
XX, YY = np.meshgrid(x, y, indexing="ij")

phi_pred_all = np.zeros((nt, nx, ny), dtype=np.float64)
Ts_pred_all  = np.zeros((nt, nx, ny), dtype=np.float64)
Tf_pred_all  = np.zeros((nt, nx, ny), dtype=np.float64)

for k, tk in enumerate(time_arr):
    TT = np.full_like(XX, float(tk), dtype=np.float64)
    X_eval = np.stack([XX.reshape(-1), YY.reshape(-1), TT.reshape(-1)], axis=1)
    pred = np.array(mlp_apply(params, jnp.array(X_eval, dtype=DTYPE)))

    phi_pred_all[k] = pred[:, 0].reshape(nx, ny)
    Ts_pred_all[k]  = pred[:, 1].reshape(nx, ny)
    Tf_pred_all[k]  = pred[:, 2].reshape(nx, ny)

# ============================================================
# masks
# ============================================================
fuel_mask = x < float(Ls) - 1e-12
cool_mask = x > float(Ls) + 1e-12

# ============================================================
# space-time relative L2
# ============================================================
def rel_l2_spacetime(pred_arr, true_arr):
    pred_arr = np.asarray(pred_arr, dtype=np.float64)
    true_arr = np.asarray(true_arr, dtype=np.float64)
    return np.sqrt(np.mean((pred_arr - true_arr)**2) / (np.mean(true_arr**2) + 1e-30))

phi_rel_st = rel_l2_spacetime(
    phi_pred_all[:, fuel_mask, :],
    phi_ref[:, fuel_mask, :]
)

Ts_rel_st = rel_l2_spacetime(
    Ts_pred_all[:, fuel_mask, :],
    T_ref[:, fuel_mask, :]
)

Tf_rel_st = rel_l2_spacetime(
    Tf_pred_all[:, cool_mask, :],
    T_ref[:, cool_mask, :]
)

print("SPACE-TIME RELATIVE L2")
print(f"phi (fuel): {phi_rel_st:.6e}")
print(f"Ts  (fuel): {Ts_rel_st:.6e}")
print(f"Tf  (cool): {Tf_rel_st:.6e}")

SPACE-TIME RELATIVE L2
phi (fuel): 3.980897e-03
Ts  (fuel): 5.097354e-02
Tf  (cool): 3.095380e-01


In [0]:
SPACE-TIME RELATIVE L2
phi (fuel): 3.445764e-03
Ts  (fuel): 5.021142e-02
Tf  (cool): 2.902433e-01

In [2]:
def print_range(name, pred, true):
    print(f"{name}:")
    print(f"  pred min/max: {pred.min():.4f} / {pred.max():.4f}")
    print(f"  true min/max: {true.min():.4f} / {true.max():.4f}")

print("\n===== RANGE CHECK =====")

print_range("phi (fuel)",
            phi_pred_all[:, fuel_mask, :],
            phi_ref[:, fuel_mask, :])

print_range("Ts (fuel)",
            Ts_pred_all[:, fuel_mask, :],
            T_ref[:, fuel_mask, :])

print_range("Tf (cool)",
            Tf_pred_all[:, cool_mask, :],
            T_ref[:, cool_mask, :])


===== RANGE CHECK =====
phi (fuel):
  pred min/max: -0.0033 / 1.1894
  true min/max: 0.0000 / 1.1919
Ts (fuel):
  pred min/max: -0.0016 / 2.0760
  true min/max: 0.0000 / 2.1983
Tf (cool):
  pred min/max: -0.3024 / 1.5298
  true min/max: 0.0000 / 2.5381


In [3]:
# find interface index
x_if = float(Ls)
ix_if = np.argmin(np.abs(x - x_if))

print("\n===== INTERFACE TEMPERATURE CHECK =====")

for k in [int(nt*0.25), int(nt*0.5), int(nt*0.75), nt-1]:
    Ts_left  = Ts_pred_all[k, ix_if-1, :]
    Tf_right = Tf_pred_all[k, ix_if+1, :]

    diff = Ts_left - Tf_right

    print(f"t = {time_arr[k]:.3f}")
    print(f"  mean |Ts-Tf| = {np.mean(np.abs(diff)):.4e}")
    print(f"  max  |Ts-Tf| = {np.max(np.abs(diff)):.4e}")


===== INTERFACE TEMPERATURE CHECK =====
t = 0.250
  mean |Ts-Tf| = 2.5646e-03
  max  |Ts-Tf| = 1.7170e-02
t = 0.500
  mean |Ts-Tf| = 1.0488e-02
  max  |Ts-Tf| = 1.6330e-01
t = 0.750
  mean |Ts-Tf| = 2.0127e-02
  max  |Ts-Tf| = 3.3009e-01
t = 1.000
  mean |Ts-Tf| = 2.1019e-02
  max  |Ts-Tf| = 4.8945e-02


In [4]:
import numpy as np

# constants
D_phi = 0.05
Sigma_a = 1.0
k_s = 0.50
k_f = 0.15
gamma = 10.0
vy = 1.0
Ls_float = 0.4

# masks
fuel_mask = x < Ls_float - 1e-12
cool_mask = x > Ls_float + 1e-12

# avoid boundary points for finite-difference residual check
fuel_x_idx = np.where(fuel_mask)[0]
cool_x_idx = np.where(cool_mask)[0]

# remove near-boundary indices
fuel_x_idx = fuel_x_idx[2:-2]
cool_x_idx = cool_x_idx[2:-2]
y_idx = np.arange(2, len(y)-2)
t_idx = np.arange(2, len(time_arr)-2)

# finite differences from MOOSE truth
# arrays shape: (nt, nx, ny)
phi = phi_ref
T = T_ref

phi_t = np.gradient(phi, time_arr, axis=0)
phi_x = np.gradient(phi, x, axis=1)
phi_y = np.gradient(phi, y, axis=2)
phi_xx = np.gradient(phi_x, x, axis=1)
phi_yy = np.gradient(phi_y, y, axis=2)

T_t = np.gradient(T, time_arr, axis=0)
T_x = np.gradient(T, x, axis=1)
T_y = np.gradient(T, y, axis=2)
T_xx = np.gradient(T_x, x, axis=1)
T_yy = np.gradient(T_y, y, axis=2)

# restrict interior slices
sl_fuel = np.ix_(t_idx, fuel_x_idx, y_idx)
sl_cool = np.ix_(t_idx, cool_x_idx, y_idx)

# ---- neutron PDE sign check ----
# candidate A: phi_t - D lap(phi) + Sigma phi = 0
r_phi_A = phi_t - D_phi * (phi_xx + phi_yy) + Sigma_a * phi

# candidate B: phi_t - D lap(phi) - Sigma phi = 0
r_phi_B = phi_t - D_phi * (phi_xx + phi_yy) - Sigma_a * phi

# ---- fuel heat source sign check ----
# candidate A: T_t - k_s lap(T) - gamma phi = 0
r_Ts_A = T_t - k_s * (T_xx + T_yy) - gamma * phi

# candidate B: T_t - k_s lap(T) + gamma phi = 0
r_Ts_B = T_t - k_s * (T_xx + T_yy) + gamma * phi

# ---- coolant advection sign check ----
# candidate A: T_t + vy T_y - k_f lap(T) = 0
r_Tf_A = T_t + vy * T_y - k_f * (T_xx + T_yy)

# candidate B: T_t - vy T_y - k_f lap(T) = 0
r_Tf_B = T_t - vy * T_y - k_f * (T_xx + T_yy)

def rms(a):
    return np.sqrt(np.mean(a**2))

print("\n===== MOOSE TRUTH PDE RESIDUAL SIGN CHECK =====")

print("phi residual on fuel:")
print("  A: phi_t - D lap + Sigma phi =", rms(r_phi_A[sl_fuel]))
print("  B: phi_t - D lap - Sigma phi =", rms(r_phi_B[sl_fuel]))

print("\nTs residual on fuel:")
print("  A: T_t - ks lap - gamma phi =", rms(r_Ts_A[sl_fuel]))
print("  B: T_t - ks lap + gamma phi =", rms(r_Ts_B[sl_fuel]))

print("\nTf residual on coolant:")
print("  A: T_t + vy Ty - kf lap =", rms(r_Tf_A[sl_cool]))
print("  B: T_t - vy Ty - kf lap =", rms(r_Tf_B[sl_cool]))


===== MOOSE TRUTH PDE RESIDUAL SIGN CHECK =====
phi residual on fuel:
  A: phi_t - D lap + Sigma phi = 0.0035883818601430513
  B: phi_t - D lap - Sigma phi = 0.7988460027391134

Ts residual on fuel:
  A: T_t - ks lap - gamma phi = 0.6706444029503995
  B: T_t - ks lap + gamma phi = 8.010478306728869

Tf residual on coolant:
  A: T_t + vy Ty - kf lap = 0.2959728990926346
  B: T_t - vy Ty - kf lap = 2.2892944971264435


In [5]:
from jax import jacrev, vmap

x_if = float(Ls)

def predict_batch(X_np, batch=20000):
    outs = []
    for i in range(0, len(X_np), batch):
        Xb = jnp.array(X_np[i:i+batch], dtype=DTYPE)
        outs.append(np.array(mlp_apply(params, Xb)))
    return np.vstack(outs)

print("\n===== EXACT NN INTERFACE TEMPERATURE CHECK =====")

for tq in [0.25, 0.50, 0.75, 1.00]:
    k = int(np.argmin(np.abs(time_arr - tq)))
    t_used = float(time_arr[k])

    X_if_eval = np.stack([
        np.full_like(y, x_if, dtype=np.float64),
        y,
        np.full_like(y, t_used, dtype=np.float64),
    ], axis=1)

    pred_if = predict_batch(X_if_eval)
    phi_if = pred_if[:, 0]
    Ts_if  = pred_if[:, 1]
    Tf_if  = pred_if[:, 2]

    diff = Ts_if - Tf_if

    ix_if = int(np.argmin(np.abs(x - x_if)))
    T_true_if = T_ref[k, ix_if, :]

    print(f"t = {t_used:.3f}")
    print(f"  exact mean |Ts-Tf|       = {np.mean(np.abs(diff)):.4e}")
    print(f"  exact max  |Ts-Tf|       = {np.max(np.abs(diff)):.4e}")
    print(f"  mean |Ts_if - T_true_if| = {np.mean(np.abs(Ts_if - T_true_if)):.4e}")
    print(f"  mean |Tf_if - T_true_if| = {np.mean(np.abs(Tf_if - T_true_if)):.4e}")
    print(f"  Ts_if range: {Ts_if.min():.4f} / {Ts_if.max():.4f}")
    print(f"  Tf_if range: {Tf_if.min():.4f} / {Tf_if.max():.4f}")
    print(f"  true  range: {T_true_if.min():.4f} / {T_true_if.max():.4f}")


===== EXACT NN INTERFACE TEMPERATURE CHECK =====
t = 0.250
  exact mean |Ts-Tf|       = 4.0104e-04
  exact max  |Ts-Tf|       = 1.3281e-02
  mean |Ts_if - T_true_if| = 3.8085e-03
  mean |Tf_if - T_true_if| = 3.5086e-03
  Ts_if range: 0.0657 / 0.0882
  Tf_if range: 0.0524 / 0.0881
  true  range: 0.0000 / 0.0975
t = 0.500
  exact mean |Ts-Tf|       = 2.7900e-03
  exact max  |Ts-Tf|       = 1.5155e-01
  mean |Ts_if - T_true_if| = 3.1825e-02
  mean |Tf_if - T_true_if| = 2.9266e-02
  Ts_if range: 0.3249 / 0.4547
  Tf_if range: 0.1734 / 0.4547
  true  range: 0.0000 / 0.5474
t = 0.750
  exact mean |Ts-Tf|       = 5.8801e-03
  exact max  |Ts-Tf|       = 2.8452e-01
  mean |Ts_if - T_true_if| = 5.6419e-02
  mean |Tf_if - T_true_if| = 5.0992e-02
  Ts_if range: 0.5089 / 0.9753
  Tf_if range: 0.2244 / 0.9755
  true  range: 0.0000 / 1.2547
t = 1.000
  exact mean |Ts-Tf|       = 3.5082e-03
  exact max  |Ts-Tf|       = 2.1439e-02
  mean |Ts_if - T_true_if| = 1.7676e-01
  mean |Tf_if - T_true_if| = 1.

In [6]:
def forward_one(z):
    return mlp_apply(params, z[None, :])[0, :]

print("\n===== EXACT NN INTERFACE FLUX CHECK =====")

for tq in [0.25, 0.50, 0.75, 1.00]:
    k = int(np.argmin(np.abs(time_arr - tq)))
    t_used = float(time_arr[k])

    X_if_eval = np.stack([
        np.full_like(y, x_if, dtype=np.float64),
        y,
        np.full_like(y, t_used, dtype=np.float64),
    ], axis=1)

    J_if = np.array(vmap(jacrev(forward_one))(jnp.array(X_if_eval, dtype=DTYPE)))

    Ts_x = J_if[:, 1, 0]
    Tf_x = J_if[:, 2, 0]

    flux_jump = -0.50 * Ts_x + 0.15 * Tf_x

    print(f"t = {t_used:.3f}")
    print(f"  mean |flux jump| = {np.mean(np.abs(flux_jump)):.4e}")
    print(f"  max  |flux jump| = {np.max(np.abs(flux_jump)):.4e}")


===== EXACT NN INTERFACE FLUX CHECK =====
t = 0.250
  mean |flux jump| = 3.6014e-04
  max  |flux jump| = 2.9954e-03
t = 0.500
  mean |flux jump| = 6.7401e-03
  max  |flux jump| = 3.4940e-01
t = 0.750
  mean |flux jump| = 3.5040e-03
  max  |flux jump| = 8.7193e-02
t = 1.000
  mean |flux jump| = 8.2016e-03
  max  |flux jump| = 5.8635e-02


In [7]:
# ===== Compare predicted interface flux magnitude to MOOSE truth =====

from jax import jacrev, vmap

def forward_one(z):
    return mlp_apply(params, z[None, :])[0, :]

x_if = float(Ls)
ix_if = np.argmin(np.abs(x - x_if))

ks_val = 0.50
kf_val = 0.15

print("\n===== INTERFACE FLUX MAGNITUDE CHECK =====")

for tq in [0.25, 0.50, 0.75, 1.00]:
    k = int(np.argmin(np.abs(time_arr - tq)))
    t_used = float(time_arr[k])

    X_if_eval = np.stack([
        np.full_like(y, x_if, dtype=np.float64),
        y,
        np.full_like(y, t_used, dtype=np.float64),
    ], axis=1)

    J_if = np.array(vmap(jacrev(forward_one))(jnp.array(X_if_eval, dtype=DTYPE)))

    Ts_x = J_if[:, 1, 0]
    Tf_x = J_if[:, 2, 0]

    # predicted interface fluxes
    q_s_pred = -ks_val * Ts_x
    q_f_pred = -kf_val * Tf_x

    # MOOSE one-sided approximate fuel-side flux
    # interface node ix_if, one fuel node left ix_if-1
    dx_left = x[ix_if] - x[ix_if - 1]
    T_if_true = T_ref[k, ix_if, :]
    T_left_true = T_ref[k, ix_if - 1, :]

    T_x_true_left = (T_if_true - T_left_true) / dx_left
    q_true = -ks_val * T_x_true_left

    print(f"t = {t_used:.3f}")
    print(f"  pred q_s mean/max abs: {np.mean(np.abs(q_s_pred)):.4e} / {np.max(np.abs(q_s_pred)):.4e}")
    print(f"  pred q_f mean/max abs: {np.mean(np.abs(q_f_pred)):.4e} / {np.max(np.abs(q_f_pred)):.4e}")
    print(f"  true q   mean/max abs: {np.mean(np.abs(q_true)):.4e} / {np.max(np.abs(q_true)):.4e}")
    print(f"  rel q_s error mean: {np.mean(np.abs(q_s_pred - q_true)) / (np.mean(np.abs(q_true)) + 1e-12):.4e}")


===== INTERFACE FLUX MAGNITUDE CHECK =====
t = 0.250
  pred q_s mean/max abs: 1.2312e-01 / 2.1322e-01
  pred q_f mean/max abs: 1.2326e-01 / 2.1023e-01
  true q   mean/max abs: 1.3555e-01 / 5.0208e+00
  rel q_s error mean: 1.4195e-01
t = 0.500
  pred q_s mean/max abs: 4.3057e-01 / 9.2996e-01
  pred q_f mean/max abs: 4.2441e-01 / 7.3468e-01
  true q   mean/max abs: 4.7664e-01 / 2.2141e+01
  rel q_s error mean: 1.9708e-01
t = 0.750
  pred q_s mean/max abs: 7.8885e-01 / 2.5463e+00
  pred q_f mean/max abs: 7.8687e-01 / 2.4591e+00
  true q   mean/max abs: 8.3839e-01 / 4.3047e+01
  rel q_s error mean: 2.6469e-01
t = 1.000
  pred q_s mean/max abs: 9.6575e-01 / 1.7094e+00
  pred q_f mean/max abs: 9.5794e-01 / 1.6909e+00
  true q   mean/max abs: 1.1762e+00 / 6.3356e+01
  rel q_s error mean: 3.1865e-01


In [14]:
def rel_l2_spacetime_np(pred, true):
    return np.sqrt(np.mean((pred - true)**2) / (np.mean(true**2) + 1e-30))

def affine_fit_error(pred, true, name):
    p = np.asarray(pred, dtype=np.float64).reshape(-1)
    q = np.asarray(true, dtype=np.float64).reshape(-1)

    A = np.vstack([p, np.ones_like(p)]).T
    a, b = np.linalg.lstsq(A, q, rcond=None)[0]

    q_fit = a * p + b

    before = rel_l2_spacetime_np(p, q)
    after = rel_l2_spacetime_np(q_fit, q)

    print(f"\n{name}")
    print(f"  best affine true ≈ a*pred+b: a={a:.6f}, b={b:.6f}")
    print(f"  rel L2 before: {before:.6e}")
    print(f"  rel L2 after : {after:.6e}")

affine_fit_error(
    Ts_pred_all[:, fuel_mask, :],
    T_ref[:, fuel_mask, :],
    "Ts fuel"
)

affine_fit_error(
    Tf_pred_all[:, cool_mask, :],
    T_ref[:, cool_mask, :],
    "Tf coolant"
)


Ts fuel
  best affine true ≈ a*pred+b: a=1.574708, b=0.170742
  rel L2 before: 5.035012e-01
  rel L2 after : 2.144305e-01

Tf coolant
  best affine true ≈ a*pred+b: a=2.289357, b=0.088737
  rel L2 before: 7.108456e-01
  rel L2 after : 3.928582e-01


In [15]:
def rel_l2_np(pred, true):
    return np.linalg.norm(pred - true) / (np.linalg.norm(true) + 1e-12)

print("\n===== TIME-WISE RELATIVE L2 =====")
for k in range(nt):
    phi_e = rel_l2_np(phi_pred_all[k, fuel_mask, :], phi_ref[k, fuel_mask, :])
    Ts_e  = rel_l2_np(Ts_pred_all[k, fuel_mask, :], T_ref[k, fuel_mask, :])
    Tf_e  = rel_l2_np(Tf_pred_all[k, cool_mask, :], T_ref[k, cool_mask, :])
    if k % max(1, nt // 10) == 0 or k == nt - 1:
        print(f"t={time_arr[k]:.4f}: phi={phi_e:.3e}, Ts={Ts_e:.3e}, Tf={Tf_e:.3e}")


===== TIME-WISE RELATIVE L2 =====
t=0.0000: phi=1.150e+11, Ts=1.314e+10, Tf=1.929e+10
t=0.1000: phi=7.138e-01, Ts=3.363e-01, Tf=4.181e+00
t=0.2000: phi=1.074e+00, Ts=4.685e-01, Tf=1.952e+00
t=0.3000: phi=1.235e+00, Ts=5.730e-01, Tf=1.467e+00
t=0.4000: phi=1.101e+00, Ts=6.273e-01, Tf=1.200e+00
t=0.5000: phi=7.805e-01, Ts=6.198e-01, Tf=9.881e-01
t=0.6000: phi=5.822e-01, Ts=5.737e-01, Tf=8.336e-01
t=0.7000: phi=6.112e-01, Ts=5.312e-01, Tf=7.488e-01
t=0.8000: phi=6.100e-01, Ts=4.992e-01, Tf=7.015e-01
t=0.9000: phi=5.270e-01, Ts=4.689e-01, Tf=6.710e-01
t=1.0000: phi=4.335e-01, Ts=4.367e-01, Tf=6.519e-01


In [10]:
print("\n===== WORST Tf ERROR LOCATION =====")
err_Tf = np.abs(Tf_pred_all[:, cool_mask, :] - T_ref[:, cool_mask, :])
idx = np.unravel_index(np.argmax(err_Tf), err_Tf.shape)

cool_x = x[cool_mask]
kt, ix_c, iy = idx

print("max |Tf error|:", err_Tf[idx])
print("at t =", time_arr[kt])
print("at x =", cool_x[ix_c])
print("at y =", y[iy])
print("Tf_pred =", Tf_pred_all[:, cool_mask, :][idx])
print("Tf_true =", T_ref[:, cool_mask, :][idx])


===== WORST Tf ERROR LOCATION =====
max |Tf error|: 1.4842186020142085
at t = 1.0
at x = 0.541015625
at y = 1.0
Tf_pred = 0.9017983894894582
Tf_true = 2.386016991503667


In [13]:
print("\n===== COOLANT TOP VALUE CHECK =====")

jy_top = len(y) - 1
jy_mid = np.argmin(np.abs(y - 0.5))
jy_low = np.argmin(np.abs(y - 0.2))

for tq in [0.25, 0.50, 0.75, 1.00]:
    k = int(np.argmin(np.abs(time_arr - tq)))
    print(f"\nt = {time_arr[k]:.3f}")

    for label, jy in [("low y=0.2", jy_low), ("mid y=0.5", jy_mid), ("top y=1.0", jy_top)]:
        pred_mean = np.mean(Tf_pred_all[k, cool_mask, jy])
        true_mean = np.mean(T_ref[k, cool_mask, jy])
        pred_max = np.max(Tf_pred_all[k, cool_mask, jy])
        true_max = np.max(T_ref[k, cool_mask, jy])

        print(
            f"  {label}: "
            f"pred mean/max = {pred_mean:.4f}/{pred_max:.4f}, "
            f"true mean/max = {true_mean:.4f}/{true_max:.4f}, "
            f"diff mean = {pred_mean - true_mean:.4f}"
        )


===== COOLANT TOP VALUE CHECK =====

t = 0.250
  low y=0.2: pred mean/max = -0.0101/0.0010, true mean/max = 0.0122/0.0798, diff mean = -0.0223
  mid y=0.5: pred mean/max = -0.0204/-0.0002, true mean/max = 0.0151/0.0890, diff mean = -0.0355
  top y=1.0: pred mean/max = 0.0108/0.0676, true mean/max = 0.0236/0.0989, diff mean = -0.0128

t = 0.500
  low y=0.2: pred mean/max = -0.0290/-0.0025, true mean/max = 0.0759/0.3766, diff mean = -0.1049
  mid y=0.5: pred mean/max = -0.0324/-0.0078, true mean/max = 0.1124/0.4471, diff mean = -0.1447
  top y=1.0: pred mean/max = 0.0549/0.2754, true mean/max = 0.2405/0.5834, diff mean = -0.1857

t = 0.750
  low y=0.2: pred mean/max = 0.0185/0.1354, true mean/max = 0.1721/0.7543, diff mean = -0.1536
  mid y=0.5: pred mean/max = 0.0571/0.3254, true mean/max = 0.2866/0.9380, diff mean = -0.2295
  top y=1.0: pred mean/max = 0.1436/0.5699, true mean/max = 0.8005/1.4246, diff mean = -0.6569

t = 1.000
  low y=0.2: pred mean/max = 0.0993/0.4556, true mean/max